# Детекция автотранспорта — обучение YOLOv8 (n / s / m)

Выпускной проект DLS, продуктовый трек. Задача: детекция автотранспорта и других участников
дорожного движения на кадрах с дронов (10 классов: пешеход, человек, велосипед, машина,
фургон, грузовик, рикша, рикша с навесом, автобус, мотоцикл) на датасете
[VisDrone2019-DET](https://www.kaggle.com/datasets/kushagrapandya/visdrone-dataset)
(Tianjin University, 10209 изображений, готовая разбивка train/val/test).

В этом ноутбуке:
1. Загрузка датасета с Kaggle через `kagglehub`.
2. Конвертация разметки VisDrone → YOLO.
3. Обучение трёх моделей: **YOLOv8n**, **YOLOv8s**, **YOLOv8m**.
4. Сравнение метрик (mAP@0.5, mAP@0.5:0.95, скорость обучения/инференса).
5. Экспорт весов и загрузка на Hugging Face Hub для использования в веб-приложении.

**Важно:** для скачивания датасета нужен Kaggle API-токен — создайте его на
[kaggle.com/settings](https://www.kaggle.com/settings) (API → Create New Token) и загрузите
`kaggle.json` в ячейке ниже.

## 1. Установка зависимостей и авторизация Kaggle

In [ ]:
!pip install -q ultralytics==8.3.63 kagglehub huggingface_hub

In [ ]:
from google.colab import files
import os

# Загрузите kaggle.json, скачанный с https://www.kaggle.com/settings
os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # выберите kaggle.json
for fname in uploaded:
    os.rename(fname, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

## 2. Загрузка датасета VisDrone2019-DET

In [ ]:
import kagglehub

DATASET_ROOT = kagglehub.dataset_download("kushagrapandya/visdrone-dataset")
print("Датасет скачан в:", DATASET_ROOT)
print("Содержимое:", os.listdir(DATASET_ROOT))

## 3. Конвертация разметки VisDrone → YOLO

Формат аннотаций VisDrone — построчный `.txt` для каждого изображения:
`x, y, w, h, score, category, truncation, occlusion` в абсолютных пиксельных координатах.
Категория `0` — служебный класс "ignored regions", исключаем его. Остальные категории
(`1`..`10`) сдвигаем на единицу, чтобы получить YOLO-индексы `0`..`9`.

In [ ]:
CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
]


SPLIT_DIRS = {
    "train": "VisDrone2019-DET-train/VisDrone2019-DET-train",
    "val": "VisDrone2019-DET-val/VisDrone2019-DET-val",
    "test": "VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev",
}

for split, dirname in SPLIT_DIRS.items():
    split_path = os.path.join(DATASET_ROOT, dirname)
    assert os.path.isdir(split_path), f"Не найдена папка: {split_path}"
    n_images = len(os.listdir(os.path.join(split_path, "images")))
    print(f"{split}: {n_images} изображений ({split_path})")

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm


YOLO_DATASET_ROOT = Path("/content/yolo_dataset")


def convert_box(size, box):
    # Перевод VisDrone bbox (x, y, w, h абсолютные) в YOLO xywh (нормализованные, центр)
    dw, dh = 1.0 / size[0], 1.0 / size[1]
    x, y, w, h = box
    return (x + w / 2) * dw, (y + h / 2) * dh, w * dw, h * dh


def visdrone_to_yolo(split: str, src_dir: Path):
    dst_dir = YOLO_DATASET_ROOT / split
    dst_dir.mkdir(parents=True, exist_ok=True)

    images_link = dst_dir / "images"
    if not images_link.exists():
        images_link.symlink_to(src_dir / "images")

    labels_dir = dst_dir / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)

    ann_files = list((src_dir / "annotations").glob("*.txt"))
    for ann_path in tqdm(ann_files, desc=f"Конвертация {split}"):
        img_path = (src_dir / "images" / ann_path.name).with_suffix(".jpg")
        img_size = Image.open(img_path).size

        lines = []
        with open(ann_path) as f:
            for row in f.read().strip().splitlines():
                parts = row.split(",")
                if parts[4] == "0":  # score=0 -> ignored region, пропускаем
                    continue
                cls_idx = int(parts[5]) - 1  # VisDrone категории 1..10 -> YOLO 0..9
                if not (0 <= cls_idx < len(CLASS_NAMES)):
                    continue
                box = convert_box(img_size, tuple(map(int, parts[:4])))
                lines.append(f"{cls_idx} " + " ".join(f"{v:.6f}" for v in box))

        with open(labels_dir / ann_path.with_suffix(".txt").name, "w") as f:
            f.write("\n".join(lines))


for split, dirname in SPLIT_DIRS.items():
    visdrone_to_yolo(split, Path(DATASET_ROOT) / dirname)

In [ ]:
dataset_yaml = f"""path: {YOLO_DATASET_ROOT}
train: train
val: val
test: test

names:
{chr(10).join(f'  {idx}: {name}' for idx, name in enumerate(CLASS_NAMES))}
"""

DATASET_YAML_PATH = "/content/vehicle_detection.yaml"
with open(DATASET_YAML_PATH, "w", encoding="utf-8") as f:
    f.write(dataset_yaml)

print(dataset_yaml)

## 4. Обучение моделей

Обучаем три модели одного семейства YOLOv8 разного размера — **n** (nano), **s** (small),
**m** (medium). Даём пользователю в приложении выбор между быстрой/лёгкой и точной/медленной
моделью.

In [ ]:
from ultralytics import YOLO
import time

EPOCHS = 60
IMG_SIZE = 640
RUNS = {}

In [ ]:
labels_dir = YOLO_DATASET_ROOT / "train" / "labels"
label_files = list(labels_dir.glob("*.txt"))
print("Всего label-файлов:", len(label_files))

non_empty = [f for f in label_files if f.stat().st_size > 0]
print("Непустых файлов:", len(non_empty))

# Посмотрим на тот же файл, что мы проверяли вручную
sample_label = labels_dir / "0000325_04801_d_0000686.txt"
print("Существует:", sample_label.exists())
if sample_label.exists():
    print("Размер:", sample_label.stat().st_size, "байт")
    print("Содержимое:")
    print(sample_label.read_text())

### 4.1 YOLOv8n

In [ ]:


print(open("/content/vehicle_detection.yaml").read())

In [ ]:
model_n = YOLO("yolov8n.pt")
start = time.time()
results_n = model_n.train(
    data=DATASET_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=16,
    name="yolov8n_vehicle_detection",
    patience=15,
)
RUNS["yolov8n"] = {"train_time_sec": time.time() - start, "model": model_n}

### 4.2 YOLOv8s

In [ ]:
model_s = YOLO("yolov8s.pt")
start = time.time()
results_s = model_s.train(
    data=DATASET_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=16,
    name="yolov8s_vehicle_detection",
    patience=15,
)
RUNS["yolov8s"] = {"train_time_sec": time.time() - start, "model": model_s}

### 4.3 YOLOv8m

In [ ]:
model_m = YOLO("yolov8m.pt")
start = time.time()
results_m = model_m.train(
    data=DATASET_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=8,
    name="yolov8m_vehicle_detection",
    patience=15,
)
RUNS["yolov8m"] = {"train_time_sec": time.time() - start, "model": model_m}

## 5. Оценка на тестовой выборке и сравнение метрик

In [ ]:
metrics_summary = {}

for model_id in ("yolov8n", "yolov8s", "yolov8m"):
    model = RUNS[model_id]["model"]
    val_results = model.val(data=DATASET_YAML_PATH, split="test", imgsz=IMG_SIZE)
    metrics_summary[model_id] = {
        "mAP50": float(val_results.box.map50),
        "mAP50-95": float(val_results.box.map),
        "precision": float(val_results.box.mp),
        "recall": float(val_results.box.mr),
        "train_time_sec": RUNS[model_id]["train_time_sec"],
    }

import pandas as pd
metrics_df = pd.DataFrame(metrics_summary).T
metrics_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_df["mAP50"].plot(kind="bar", ax=axes[0], color=["#e74c3c", "#3498db", "#2ecc71"])
axes[0].set_title("mAP@0.5 по моделям")
axes[0].set_ylabel("mAP@0.5")

metrics_df["mAP50-95"].plot(kind="bar", ax=axes[1], color=["#e74c3c", "#3498db", "#2ecc71"])
axes[1].set_title("mAP@0.5:0.95 по моделям")
axes[1].set_ylabel("mAP@0.5:0.95")

(metrics_df["train_time_sec"] / 60).plot(kind="bar", ax=axes[2], color=["#e74c3c", "#3498db", "#2ecc71"])
axes[2].set_title("Время обучения (мин)")
axes[2].set_ylabel("Минуты")

plt.tight_layout()
plt.savefig("/content/metrics_comparison.png", dpi=150)
plt.show()

### Кривые обучения

Ultralytics сам сохраняет `results.png` (loss/mAP по эпохам) в папку `runs/detect/<name>/`
для каждого запуска. Отобразим их рядом для сравнения скорости сходимости.

In [ ]:
from PIL import Image as PILImage

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
for ax, model_id in zip(axes, ("yolov8n", "yolov8s", "yolov8m")):
    results_png = f"runs/detect/{model_id}_vehicle_detection/results.png"
    if os.path.exists(results_png):
        ax.imshow(PILImage.open(results_png))
        ax.set_title(f"Кривые обучения: {model_id}")
        ax.axis("off")
plt.tight_layout()
plt.savefig("/content/training_curves_all.png", dpi=150)
plt.show()

## 6. Анализ результатов и гипотезы

Заполните после просмотра метрик и графиков (пример структуры анализа):

- **Какая модель быстрее обучилась?** Сравните `train_time_sec` — обычно YOLOv8n быстрее
  всего благодаря меньшему числу параметров, YOLOv8m — медленнее.
- **Какая модель точнее (mAP@0.5)?** Обычно YOLOv8m > YOLOv8s > YOLOv8n за счёт большей
  ёмкости модели, но прирост точности может быть непропорционален приросту времени обучения
  и инференса.
- **Гипотезы по возможным проблемам:**
  - Сильный дисбаланс классов (`car` — самый частый класс, `awning-tricycle` и `tricycle` —
    редкие) может приводить к низкому recall на редких классах.
  - Мелкие объекты на аэрофото (пешеходы, велосипеды с большой высоты съёмки) сложнее
    детектировать — типичная проблема VisDrone из-за высокого разрешения кадров и малого
    относительного размера объектов.
  - Похожие по силуэту классы (`car` vs `van`, `truck` vs `bus`, `tricycle` vs
    `awning-tricycle`) могут путаться между собой — видно по confusion matrix
    (`runs/detect/<name>/confusion_matrix.png`).
  - Загущённые сцены (много мелких объектов рядом) — модель может пропускать перекрывающиеся
    объекты (низкий recall при высокой occlusion).
- **Как улучшить:**
  - Увеличение разрешения (`imgsz`) для лучшей детекции мелких объектов.
  - Аугментации, ориентированные на редкие классы (oversampling `tricycle`, `awning-tricycle`).
  - Tiled inference / slicing (например, SAHI) для аэрофото с мелкими объектами.
  - Больше эпох / learning rate scheduling, если кривые обучения ещё не вышли на плато.

## 7. Экспорт весов и загрузка на Hugging Face Hub

In [ ]:
import shutil

MODELS_EXPORT_DIR = "/content/exported_models"
os.makedirs(MODELS_EXPORT_DIR, exist_ok=True)

weight_files = {
    "yolov8n": "runs/detect/yolov8n_vehicle_detection/weights/best.pt",
    "yolov8s": "runs/detect/yolov8s_vehicle_detection/weights/best.pt",
    "yolov8m": "runs/detect/yolov8m_vehicle_detection/weights/best.pt",
}

for model_id, src in weight_files.items():
    dst = os.path.join(MODELS_EXPORT_DIR, f"{model_id}_vehicle_detection.pt")
    shutil.copy2(src, dst)
    print(f"{model_id}: {dst} ({os.path.getsize(dst) / 1e6:.1f} MB)")

In [ ]:
# Сохраняем итоговые метрики рядом с весами
metrics_df.to_csv(os.path.join(MODELS_EXPORT_DIR, "metrics_summary.csv"))
metrics_df

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # вставьте токен с правами write с https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = "your-username/vehicle-detection-yolov8"

create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)

api = HfApi()
for model_id in weight_files:
    filename = f"{model_id}_vehicle_detection.pt"
    api.upload_file(
        path_or_fileobj=os.path.join(MODELS_EXPORT_DIR, filename),
        path_in_repo=filename,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"Загружено: {filename}")

print(f"\nВеса доступны на https://huggingface.co/{HF_REPO_ID}")
print(f"Задайте переменную окружения HF_REPO_ID={HF_REPO_ID} в backend/frontend для авто-загрузки.")